In [ ]:
import json
import math
import random
from pathlib import Path
from typing import Optional, Tuple, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch.autograd import Function
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))

In [ ]:
LABEL2ID = {'negative': 0, 'positive': 1}
ID2LABEL = {0: 'negative', 1: 'positive'}

def _resolve_text_column(df: pd.DataFrame) -> str:
    for col in ['content', 'reviewContent', 'text', 'review']:
        if col in df.columns:
            return col
    raise ValueError(f'Cannot find text column. Available columns: {df.columns.tolist()}')

def _resolve_label_column(df: pd.DataFrame) -> str:
    for col in ['label', 'sentiment', 'target']:
        if col in df.columns:
            return col
    raise ValueError(f'Cannot find label column. Available columns: {df.columns.tolist()}')

def _normalize_labels(series: pd.Series) -> pd.Series:
    normalized = series.astype(str).str.strip().str.lower()
    return normalized.replace({'neg': 'negative', 'pos': 'positive'})

def clean_dataframe(df: pd.DataFrame, text_col: str, label_col: Optional[str] = None) -> pd.DataFrame:
    use_cols = [text_col] + ([label_col] if label_col else [])
    out = df[use_cols].copy()
    out = out.dropna(subset=[text_col])
    out[text_col] = out[text_col].astype(str).str.strip()
    out = out[out[text_col] != '']

    if label_col:
        out = out.dropna(subset=[label_col])
        out[label_col] = _normalize_labels(out[label_col])
        out = out[out[label_col].isin(LABEL2ID)]

    return out.reset_index(drop=True)

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128, labels=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )

        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

def make_loaders(
    tokenizer,
    source_path: Path,
    target_path: Path,
    eval_path: Optional[Path],
    batch_size: int,
    max_length: int,
    seed: int,
) -> Tuple[DataLoader, DataLoader, DataLoader, Optional[DataLoader], dict]:
    source_df = pd.read_csv(source_path)
    source_text_col = _resolve_text_column(source_df)
    source_label_col = _resolve_label_column(source_df)
    source_df = clean_dataframe(source_df, source_text_col, source_label_col)
    source_df['label_id'] = source_df[source_label_col].map(LABEL2ID)

    train_df, val_df = train_test_split(
        source_df,
        test_size=0.1,
        random_state=seed,
        stratify=source_df['label_id'],
    )

    target_df = pd.read_csv(target_path)
    target_text_col = _resolve_text_column(target_df)
    target_df = clean_dataframe(target_df, target_text_col)

    source_train_ds = TextDataset(
        texts=train_df[source_text_col].tolist(),
        labels=train_df['label_id'].tolist(),
        tokenizer=tokenizer,
        max_length=max_length,
    )
    source_val_ds = TextDataset(
        texts=val_df[source_text_col].tolist(),
        labels=val_df['label_id'].tolist(),
        tokenizer=tokenizer,
        max_length=max_length,
    )
    target_train_ds = TextDataset(
        texts=target_df[target_text_col].tolist(),
        labels=None,
        tokenizer=tokenizer,
        max_length=max_length,
    )

    source_train_loader = DataLoader(source_train_ds, batch_size=batch_size, shuffle=True)
    source_val_loader = DataLoader(source_val_ds, batch_size=batch_size, shuffle=False)
    target_train_loader = DataLoader(target_train_ds, batch_size=batch_size, shuffle=True)

    eval_loader = None
    eval_samples = 0

    if eval_path is not None and eval_path.exists():
        eval_df = pd.read_csv(eval_path)
        eval_text_col = _resolve_text_column(eval_df)
        eval_label_col = _resolve_label_column(eval_df)

        if 'origin' in eval_df.columns:
            origin = eval_df['origin'].astype(str).str.lower()
            target_only = eval_df[origin == 'target']
            if len(target_only) > 0:
                eval_df = target_only

        eval_df = clean_dataframe(eval_df, eval_text_col, eval_label_col)
        eval_df['label_id'] = eval_df[eval_label_col].map(LABEL2ID)

        eval_ds = TextDataset(
            texts=eval_df[eval_text_col].tolist(),
            labels=eval_df['label_id'].tolist(),
            tokenizer=tokenizer,
            max_length=max_length,
        )
        eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False)
        eval_samples = len(eval_ds)

    stats = {
        'source_train': len(source_train_ds),
        'source_val': len(source_val_ds),
        'target_train': len(target_train_ds),
        'target_eval': eval_samples,
    }

    return source_train_loader, source_val_loader, target_train_loader, eval_loader, stats

## Combined DAPT + Domain Adapter (GRL) + Task Adapter

In [ ]:
PIPE_CFG = {
    'dapt_model_dir': 'dapt_coastsent',
    'source_unlabeled_csv': 'datasets/lazada_train.csv',
    'target_unlabeled_csv': 'datasets/ID-CoastSent_preprocessed.csv',
    'source_labeled_csv': 'datasets/lazada_train.csv',
    'target_eval_csv': 'datasets/lazada_test.csv',
    'output_dir': 'outputs/adapter_stack',
    'batch_size': 32,
    'max_length': 128,
    'epochs_domain': 3,
    'epochs_task': 15,
    'learning_rate': 1e-5,
    'domain_adapter_lr': 5e-5,
    'task_adapter_lr': 1e-4,
    'weight_decay': 0.05,
    'warmup_ratio': 0.1,
    'dropout': 0.2,
    'domain_adapter_dim': 16,
    'task_adapter_dim': 16,
    'max_grad_norm': 1.0,
    'unfreeze_top_layers': 3,
    'seed': 42,
}

In [ ]:
class UnlabeledTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }

def make_unlabeled_loaders(
    tokenizer,
    source_path: Path,
    target_path: Path,
    batch_size: int,
    max_length: int,
    seed: int,
    ) -> Tuple[DataLoader, DataLoader]:
    source_df = pd.read_csv(source_path)
    source_text_col = _resolve_text_column(source_df)
    source_df = clean_dataframe(source_df, source_text_col)

    target_df = pd.read_csv(target_path)
    target_text_col = _resolve_text_column(target_df)
    target_df = clean_dataframe(target_df, target_text_col)

    source_ds = UnlabeledTextDataset(
        texts=source_df[source_text_col].tolist(),
        tokenizer=tokenizer,
        max_length=max_length,
    )
    target_ds = UnlabeledTextDataset(
        texts=target_df[target_text_col].tolist(),
        tokenizer=tokenizer,
        max_length=max_length,
    )

    g = torch.Generator()
    g.manual_seed(seed)

    source_loader = DataLoader(source_ds, batch_size=batch_size, shuffle=True, generator=g)
    target_loader = DataLoader(target_ds, batch_size=batch_size, shuffle=True, generator=g)
    return source_loader, target_loader

def make_labeled_loader(
    tokenizer,
    labeled_path: Path,
    batch_size: int,
    max_length: int,
    seed: int,
    ) -> DataLoader:
    df = pd.read_csv(labeled_path)
    text_col = _resolve_text_column(df)
    label_col = _resolve_label_column(df)
    df = clean_dataframe(df, text_col, label_col)
    df['label_id'] = df[label_col].map(LABEL2ID)

    dataset = TextDataset(
        texts=df[text_col].tolist(),
        labels=df['label_id'].tolist(),
        tokenizer=tokenizer,
        max_length=max_length,
    )
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=g)

In [ ]:
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, inputs, alpha):
        ctx.alpha = alpha
        return inputs.view_as(inputs)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None

class GradientReversalLayer(nn.Module):
    def forward(self, inputs, alpha=1.0):
        return GradientReversalFunction.apply(inputs, alpha)

In [ ]:
class PfeifferAdapter(nn.Module):
    def __init__(self, hidden_size: int, adapter_dim: int, dropout: float = 0.1):
        super().__init__()
        self.down = nn.Linear(hidden_size, adapter_dim)
        self.act = nn.ReLU()
        self.up = nn.Linear(adapter_dim, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return x + self.dropout(self.up(self.act(self.down(x))))

In [ ]:
class DomainAdapterDANN(nn.Module):
    def __init__(self, base_model_dir: str, adapter_dim: int = 16, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_dir)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.domain_adapter = PfeifferAdapter(hidden_size, adapter_dim, dropout=dropout)
        self.grl = GradientReversalLayer()
        self.domain_classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2),
        )

    def extract_features(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0]

    def forward(self, input_ids, attention_mask, alpha: float = 1.0):
        features = self.extract_features(input_ids, attention_mask)
        features = self.dropout(features)
        adapted = self.domain_adapter(features)
        reversed_features = self.grl(adapted, alpha=alpha)
        domain_logits = self.domain_classifier(reversed_features)
        return domain_logits

In [ ]:
class TaskAdapterClassifier(nn.Module):
    def __init__(self, base_model_dir: str, domain_adapter_dim: int, task_adapter_dim: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_dir)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.domain_adapter = PfeifferAdapter(hidden_size, domain_adapter_dim, dropout=dropout)
        self.task_adapter = PfeifferAdapter(hidden_size, task_adapter_dim, dropout=dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, 2),
        )

    def extract_features(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0]

    def forward(self, input_ids, attention_mask):
        features = self.extract_features(input_ids, attention_mask)
        features = self.dropout(features)
        features = self.domain_adapter(features)
        features = self.task_adapter(features)
        return self.classifier(features)

In [ ]:
def freeze_module(module: nn.Module) -> None:
    for param in module.parameters():
        param.requires_grad = False

def unfreeze_module(module: nn.Module) -> None:
    for param in module.parameters():
        param.requires_grad = True
        
def _unfreeze_top_transformer_layers(encoder: nn.Module, num_layers: int) -> None:
    if num_layers <= 0:
        return
    if hasattr(encoder, 'encoder') and hasattr(encoder.encoder, 'layer'):
        layers = encoder.encoder.layer
        for layer in layers[-num_layers:]:
            unfreeze_module(layer)
    else:
        print('Warning: cannot find encoder layers to unfreeze; skipping.')

In [ ]:
@torch.no_grad()
def evaluate_classifier(model: TaskAdapterClassifier, dataloader: DataLoader, device_eval: torch.device):
    model.eval()
    loss_fn = nn.CrossEntropyLoss()
    all_preds, all_labels = [], []
    total_loss = 0.0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device_eval)
        attention_mask = batch['attention_mask'].to(device_eval)
        labels = batch['labels'].to(device_eval)

        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / max(1, len(dataloader))
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    return {
        'loss': avg_loss,
        'acc': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'preds': all_preds,
        'labels': all_labels,
    }

In [ ]:
def train_domain_adapter(cfg: dict) -> Path:
    set_seed(cfg['seed'])
    base_dir = cfg['dapt_model_dir']
    output_dir = Path(cfg['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    domain_adapter_path = output_dir / 'domain_adapter.pt'
    domain_classifier_path = output_dir / 'domain_classifier.pt'

    tokenizer = AutoTokenizer.from_pretrained(base_dir)
    source_loader, target_loader = make_unlabeled_loaders(
        tokenizer=tokenizer,
        source_path=Path(cfg['source_unlabeled_csv']),
        target_path=Path(cfg['target_unlabeled_csv']),
        batch_size=cfg['batch_size'],
        max_length=cfg['max_length'],
        seed=cfg['seed'],
    )

    model = DomainAdapterDANN(
        base_model_dir=base_dir,
        adapter_dim=cfg['domain_adapter_dim'],
        dropout=cfg['dropout'],
    ).to(device)

    freeze_module(model.encoder)
    unfreeze_module(model.domain_adapter)
    unfreeze_module(model.domain_classifier)

    domain_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(
        list(model.domain_adapter.parameters()) + list(model.domain_classifier.parameters()),
        lr=cfg['domain_adapter_lr'],
        weight_decay=cfg['weight_decay'],
    )

    steps_per_epoch = min(len(source_loader), len(target_loader))
    total_steps = cfg['epochs_domain'] * steps_per_epoch
    warmup_steps = int(cfg['warmup_ratio'] * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    global_step = 0
    for epoch in range(1, cfg['epochs_domain'] + 1):
        model.train()
        running_loss = 0.0
        source_iter = iter(source_loader)
        target_iter = iter(target_loader)
        progress = tqdm(range(steps_per_epoch), desc=f'Domain Epoch {epoch}/{cfg["epochs_domain"]}')
        for step_idx in progress:
            source_batch = next(source_iter)
            target_batch = next(target_iter)

            p = float(global_step) / max(1, total_steps - 1)
            alpha = 2.0 / (1.0 + math.exp(-10 * p)) - 1.0

            source_input_ids = source_batch['input_ids'].to(device)
            source_attention_mask = source_batch['attention_mask'].to(device)
            target_input_ids = target_batch['input_ids'].to(device)
            target_attention_mask = target_batch['attention_mask'].to(device)

            source_logits = model(source_input_ids, source_attention_mask, alpha=alpha)
            target_logits = model(target_input_ids, target_attention_mask, alpha=alpha)

            source_labels = torch.zeros(source_logits.size(0), dtype=torch.long, device=device)
            target_labels = torch.ones(target_logits.size(0), dtype=torch.long, device=device)

            loss_source = domain_loss_fn(source_logits, source_labels)
            loss_target = domain_loss_fn(target_logits, target_labels)
            loss = 0.5 * (loss_source + loss_target)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['max_grad_norm'])
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            global_step += 1
            progress.set_postfix(loss=f'{running_loss/(step_idx+1):.4f}', alpha=f'{alpha:.3f}')

    torch.save(model.domain_adapter.state_dict(), domain_adapter_path)
    torch.save(model.domain_classifier.state_dict(), domain_classifier_path)
    tokenizer.save_pretrained(output_dir)
    print('Saved domain adapter to:', domain_adapter_path)
    return domain_adapter_path

In [ ]:
def train_task_adapter(cfg: dict, domain_adapter_path: Path) -> Path:
    set_seed(cfg['seed'])
    base_dir = cfg['dapt_model_dir']
    output_dir = Path(cfg['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    task_adapter_path = output_dir / 'task_adapter.pt'
    task_head_path = output_dir / 'task_classifier.pt'

    tokenizer = AutoTokenizer.from_pretrained(base_dir)
    labeled_loader = make_labeled_loader(
        tokenizer=tokenizer,
        labeled_path=Path(cfg['source_labeled_csv']),
        batch_size=cfg['batch_size'],
        max_length=cfg['max_length'],
        seed=cfg['seed'],
    )

    target_eval_loader = None
    target_eval_path = cfg.get('target_eval_csv')
    if target_eval_path:
        eval_path = Path(target_eval_path)
        if eval_path.exists():
            target_eval_loader = make_labeled_loader(
                tokenizer=tokenizer,
                labeled_path=eval_path,
                batch_size=cfg['batch_size'],
                max_length=cfg['max_length'],
                seed=cfg['seed'],
            )

    model = TaskAdapterClassifier(
        base_model_dir=base_dir,
        domain_adapter_dim=cfg['domain_adapter_dim'],
        task_adapter_dim=cfg['task_adapter_dim'],
        dropout=cfg['dropout'],
    ).to(device)

    if domain_adapter_path.exists():
        model.domain_adapter.load_state_dict(torch.load(domain_adapter_path, map_location=device))
    else:
        raise FileNotFoundError(f'Domain adapter not found: {domain_adapter_path}')

    freeze_module(model.encoder)
    freeze_module(model.domain_adapter)
    unfreeze_module(model.task_adapter)
    unfreeze_module(model.classifier)
    _unfreeze_top_transformer_layers(model.encoder, int(cfg.get('unfreeze_top_layers', 0)))

    loss_fn = nn.CrossEntropyLoss()
    
    # Collect trainable parameters with per-group learning rates
    trainable_params = [
        {'params': model.task_adapter.parameters()},
        {'params': model.classifier.parameters()},
    ]
    
    # Add unfrozen encoder layers if any (with conservative LR)
    if cfg.get('unfreeze_top_layers', 0) > 0:
        unfrozen_encoder_params = [p for p in model.encoder.parameters() if p.requires_grad]
        if unfrozen_encoder_params:
            trainable_params.append({'params': unfrozen_encoder_params, 'lr': 1e-5})
    
    optimizer = AdamW(trainable_params, lr=cfg['task_adapter_lr'], weight_decay=cfg['weight_decay'])

    total_steps = cfg['epochs_task'] * len(labeled_loader)
    warmup_steps = int(cfg['warmup_ratio'] * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    for epoch in range(1, cfg['epochs_task'] + 1):
        model.train()
        running_loss = 0.0
        progress = tqdm(labeled_loader, desc=f'Task Epoch {epoch}/{cfg["epochs_task"]}')
        for step_idx, batch in enumerate(progress, 1):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg['max_grad_norm'])
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            progress.set_postfix(loss=f'{running_loss/step_idx:.4f}')

    torch.save(model.task_adapter.state_dict(), task_adapter_path)
    torch.save(model.classifier.state_dict(), task_head_path)
    tokenizer.save_pretrained(output_dir)
    print('Saved task adapter to:', task_adapter_path)

    print('\nSource labeled evaluation:')
    source_metrics = evaluate_classifier(model, labeled_loader, device)
    print(source_metrics)

    if target_eval_loader is not None:
        print('\nTarget labeled evaluation:')
        target_metrics = evaluate_classifier(model, target_eval_loader, device)
        print(target_metrics)
    else:
        print('\nTarget labeled evaluation: skipped (target_eval_csv not found)')

    return task_adapter_path

In [ ]:
domain_adapter_path = train_domain_adapter(PIPE_CFG)
task_adapter_path = train_task_adapter(PIPE_CFG, domain_adapter_path)

## Inference 

In [ ]:
def load_stacked_model(cfg: dict, device_override: Optional[torch.device] = None):
    if device_override is None:
        device_override = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    base_dir = cfg['dapt_model_dir']
    output_dir = Path(cfg['output_dir'])
    domain_adapter_path = output_dir / 'domain_adapter.pt'
    task_adapter_path = output_dir / 'task_adapter.pt'
    task_head_path = output_dir / 'task_classifier.pt'

    for path in [domain_adapter_path, task_adapter_path, task_head_path]:
        if not path.exists():
            raise FileNotFoundError(f'Missing checkpoint: {path}')

    tokenizer_dir = output_dir if (output_dir / 'tokenizer.json').exists() else base_dir
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)

    model = TaskAdapterClassifier(
        base_model_dir=base_dir,
        domain_adapter_dim=cfg['domain_adapter_dim'],
        task_adapter_dim=cfg['task_adapter_dim'],
        dropout=cfg['dropout'],
    ).to(device_override)

    model.domain_adapter.load_state_dict(torch.load(domain_adapter_path, map_location=device_override))
    model.task_adapter.load_state_dict(torch.load(task_adapter_path, map_location=device_override))
    model.classifier.load_state_dict(torch.load(task_head_path, map_location=device_override))
    model.eval()

    return model, tokenizer, device_override

@torch.no_grad()
def predict_texts_stacked(
    model: TaskAdapterClassifier,
    tokenizer,
    texts: List[str],
    device_infer: torch.device,
    max_length: int = 128,
    batch_size: int = 16,
    ):
    rows = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoding = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt',
        )

        input_ids = encoding['input_ids'].to(device_infer)
        attention_mask = encoding['attention_mask'].to(device_infer)
        logits = model(input_ids, attention_mask)
        probs = F.softmax(logits, dim=1).cpu()

        for i, text in enumerate(batch_texts):
            sentiment_id = int(torch.argmax(probs[i]).item())
            rows.append({
                'text': text,
                'sentiment_id': sentiment_id,
                'sentiment_label': ID2LABEL[sentiment_id],
                'sentiment_negative_prob': float(probs[i][0].item()),
                'sentiment_positive_prob': float(probs[i][1].item()),
            })

    return rows

def print_results(rows: List[dict], max_rows: int = 20):
    preview = rows[:max_rows]
    if not preview:
        print('No predictions.')
        return

    print(f'Showing {len(preview)} / {len(rows)} predictions')
    print('=' * 80)
    for i, row in enumerate(preview, 1):
        print(f'[{i}] sentiment={row["sentiment_label"]} (neg={row["sentiment_negative_prob"]:.4f}, pos={row["sentiment_positive_prob"]:.4f})')
        print(f'    text={row["text"]}')
        print('-' * 80)

# Example run (uncomment to execute):
model, tokenizer, infer_device = load_stacked_model(PIPE_CFG)
texts = [
    # "Produk ini kualitasnya sangat bagus, pengiriman cepat dan sesuai deskripsi.",
    # "Barang yang saya terima rusak dan tidak berfungsi dengan baik, sangat mengecewakan.",
    # "Pantainya sangat indah, airnya jernih dan suasananya tenang cocok untuk liburan.",
    "Produk ini kualitasnya sangat bagus, pengiriman cepat dan sesuai deskripsi.",
    "Barang yang saya terima rusak dan tidak berfungsi dengan baik, sangat mengecewakan.",
    "Harga cukup terjangkau tapi kualitasnya kurang memuaskan untuk pemakaian jangka panjang.",
    "Pantainya sangat indah, airnya jernih dan suasananya tenang cocok untuk liburan.",
    "Tempatnya kotor dan banyak sampah, fasilitas juga kurang terawat.",
    "Pemandangan cukup bagus tapi akses menuju lokasi cukup sulit dan jalan rusak.",
]
rows = predict_texts_stacked(
    model=model,
    tokenizer=tokenizer,
    texts=texts,
    device_infer=infer_device,
    max_length=PIPE_CFG['max_length'],
    batch_size=PIPE_CFG['batch_size'],
)
print_results(rows)
pd.DataFrame(rows)